# IDS Challenge: Teilprojekt 01 Optimierung
## Ergebnisse 

### Gruppe: The Newts
### Tutor: Henrik Wieschemeyer

## Imports & Konstanten
Grundlegende Bibliotheken und globale Parameter für die Optimierung. <br>
numpy: Array-Operationen für die Koordinatenverarbeitung beim Clustering <br>
KMeans (sklearn): geografische Gruppierung der Maschinen <br>
TIME_LIMIT = 5 * 3600: max. Tourdauer pro Roboter (5 h in Sekunden) → jede Tour muss inkl. Rückweg zum Depot darunter bleiben <br>

**calc_total_tour_time**: summiert die Fahrzeiten entlang einer Sequenz von Punkten, indem jeweils die Matrix-Zeit zwischen aufeinanderfolgenden Punkten aufaddiert wird → zentrale Hilfsfunktion zur Bewertung der Tourdauer

In [73]:
import numpy as np
from sklearn.cluster import KMeans

TIME_LIMIT = 5 * 3600

def calc_total_tour_time(sequ, matrix):
    total_time = 0
    for i in range(len(sequ) - 1):
        total_time += matrix[sequ[i], sequ[i + 1]]
    return total_time

## Datenbasisfunktionen
**Einlesen und Zusammenführen der Eingabedaten in nutzbare Datenstrukturen.** <br>

**read_ls_matrix:** liest die Distanz-/Zeitmatrix aus der Datei, speichert sie als Dictionary mit Tupel-Keys {(von, nach): zeit} → ermöglicht direkten Zugriff auf die Fahrzeit zwischen zwei beliebigen Punkten <br>
**read_points:** liest Punktdaten (Maschinen) ein → speichert je Index (x, y, stockwerk) als Koordinaten und Etage <br>
**transform_data:** lädt die Matrix und alle drei Punktdateien (gianni, lissi, ben), führt die Punkte in ein gemeinsames Dictionary zusammen → liefert points und matrix als Basis für alle weiteren Schritte


In [74]:
def read_ls_matrix(pfad="data/ls_matrix_55_42.txt"):
    matrix = {}
    with open(pfad) as datei:
        for zeile in datei:
            zeile = zeile.strip()
            if not zeile:
                continue
            teile = zeile.split(";")
            von = int(teile[0].strip())
            nach = int(teile[1].strip())
            zeit = float(teile[2].strip())
            matrix[(von, nach)] = zeit
    return matrix

def read_points(pfad):
    points = {}
    with open(pfad) as datei:
        for zeile in datei:
            zeile = zeile.strip()
            if not zeile:
                continue

            teile = zeile.split(";")
            index = int(teile[0].strip())
            x = float(teile[1].strip())
            y = float(teile[2].strip())
            stockwerk = int(teile[3])
            points[index] = (x, y, stockwerk)
    return points

def transform_data():
    matrix = read_ls_matrix("data/ls_matrix_55_42.txt")

    points_gianni = read_points("data/points_gianni_55_42.txt")
    points_lissi = read_points("data/points_lissi_55_42.txt")
    points_ben = read_points("data/points_ben_55_42.txt")

    points = {}
    points.update(points_gianni)
    points.update(points_lissi)
    points.update(points_ben)

    return points, matrix

## Clustering mit K-Means in 3 Gruppen
**Aufteilung der Maschinen auf die drei Roboter nach geografischer Nähe.** <br>
K-Means teilt die Maschinen in k=3 räumlich zusammenhängende Cluster → jeder Roboter bearbeitet einen Cluster <br>
random_state=0 sichert reproduzierbare Ergebnisse, n_init=10 mehrere Startläufe für stabileres Ergebnis <br>
Rückgabe: Liste von Clustern, jeweils mit den zugehörigen Maschinen-Indizes

**cluster_machines:** filtert das Depot (Index 0) heraus, bildet aus den (x, y)-Koordinaten der Maschinen ein Array <br>

In [75]:
def cluster_machines(points, k=3):
    maschinen_indizes = [index for index in points if index != 0]
    koordinaten = np.array([points[i][:2] for i in maschinen_indizes])
    kmeans = KMeans(n_clusters=k, random_state=0, n_init=10)
    labels = kmeans.fit_predict(koordinaten)
    clusters = [[] for _ in range(k)]
    for maschine, label in zip(maschinen_indizes, labels):
        clusters[label].append(maschine)
    return clusters

## Nutzwert & Auswahl
**Bewertung, welche Maschine als nächste in eine Tour aufgenommen werden soll.** <br>

**depot_distance:** Fahrzeit vom Depot zur Maschine → Maß für „Abgelegenheit" <br>
**insertion_cost:** Fahrzeit vom aktuellen Tourende zur Maschine → Maß für „günstig erreichbar" <br>
**machine_value:** kombiniert beide über Gewicht w: w * fern - (1-w) * guenstig → belohnt abgelegene Maschinen (lohnt sich, sie mitzunehmen), bestraft hohe Anfahrtskosten <br>
**best_value_unvisited:** durchsucht alle noch offenen Maschinen und gibt die mit dem höchsten Nutzwert zurück <br>

In [76]:
def depot_distance(machine, matrix):
    return matrix[(0, machine)]

def insertion_cost(machine, tour, matrix):
    location = tour[-1]
    return matrix[(location, machine)]

def machine_value(machine, tour, matrix, w):
    fern = depot_distance(machine, matrix)
    guenstig = insertion_cost(machine, tour, matrix)
    return w * fern - (1 - w) * guenstig

def best_value_unvisited(noch_offen, tour, matrix, w):
    best_machine = None
    best_value = None
    for machine in noch_offen:
        value = machine_value(machine, tour, matrix, w)
        if best_value is None or value > best_value:
            best_value = value
            best_machine = machine
    return best_machine

## Tour
**Aufbau einer einzelnen Tour bzw. aller Touren unter Einhaltung des Zeitlimits.** <br>

**fits_in_time:** prüft testweise, ob das Anhängen einer Maschine (inkl. Rückweg zum Depot) unter TIME_LIMIT bleibt <br>
**build_tour:** startet am Depot, wählt wiederholt per Greedy die beste Maschine; passt sie zeitlich → anhängen und aus offenen entfernen, sonst Abbruch; schließt die Tour mit Rückkehr zum Depot ab <br>
**build_all_tours:** wendet build_tour auf jeden Cluster an → liefert eine Tour pro Roboter <br>


In [77]:
def fits_in_time(tour, maschine, matrix):
    test_tour = tour + [maschine, 0]
    return calc_total_tour_time(test_tour, matrix) <= TIME_LIMIT

def build_tour(cluster, matrix, w):
    tour = [0]
    noch_offen = cluster.copy()
    while noch_offen:
        naechste = best_value_unvisited(noch_offen, tour, matrix, w)
        if fits_in_time(tour, naechste, matrix):
            tour.append(naechste)
            noch_offen.remove(naechste)
        else:
            break
    tour.append(0)
    return tour

def build_all_tours(clusters, matrix, w):
    tours = []
    for cluster in clusters:
        tour = build_tour(cluster, matrix, w)
        tours.append(tour)
    return tours
    

## Bewertung & Trade-off
**Ermittlung, wie gut eine Lösung ist bzw. welche Maschinen unbesucht bleiben.** <br>

**remaining_machines:** sammelt alle besuchten Punkte, gibt die nicht abgedeckten Maschinen zurück <br>
**sum_individual_trips:** berechnet den „Restfußweg", die Zeit, wenn jede übrige Maschine einzeln vom Depot aus per Fußweg angefahren werden müsste (Hin- und Rückweg) → Kennzahl für die Kosten der nicht abgedeckten Maschinen

In [78]:
def remaining_machines(tours, alle_maschinen):
    besucht = set()
    for tour in tours:
        for punkt in tour:
            besucht.add(punkt)
    return [m for m in alle_maschinen if m not in besucht]


def sum_individual_trips(remaining, matrix):
    summe = 0
    for maschine in remaining:
        summe += matrix[(0, maschine)]
        summe += matrix[(maschine, 0)]
    return summe



## Orchestrierung
**Optimierung des Gewichts w, das die Tourbildung steuert.** <br>
je Wert werden Touren gebaut und bewertet: score = (abdeckung, -rest_fussweg) → lexikographisch, d.h. zuerst maximale Abdeckung, bei Gleichstand minimaler Restfußweg und merkt sich die beste Kombination → gibt Touren, Abdeckung, übrige Maschinen, Restfußweg und bestes w zurück

**tune_weights:** testet systematisch w-Werte von 0.0 bis 1.0 <br>

In [79]:
def tune_weights(clusters, alle_maschinen, matrix):
    best_score = None
    best = None
    for w in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
        tours = build_all_tours(clusters, matrix, w)
        remaining = remaining_machines(tours, alle_maschinen)
        abdeckung = len(alle_maschinen) - len(remaining)
        rest_fussweg = sum_individual_trips(remaining, matrix)
        score = (abdeckung, -rest_fussweg)
        if best_score is None or score > best_score:
            best_score = score
            best = (tours, abdeckung, remaining, rest_fussweg, w)
    return best

## Ergebnis
**Aufbereitung und Ausgabe der Lösung.** <br>

**print_solution:** gibt das gewählte Gewicht, die Zahl abgedeckter Maschinen, jede Robotertour, die übrigen Maschinen sowie den Restfußweg (in s sowie h und min) aus → Zusammenfassung des Ergebnisses

In [80]:
def print_solution(tours, abdeckung, remaining, rest_fussweg, best_w):
    print(f"Beste Gewichtung w: {best_w}")
    print(f"Abgedeckte Maschinen: {abdeckung}")
    for i, tour in enumerate(tours):
        print(f"Roboter {i + 1}: {tour}")
    print(f"Übrige Maschinen ({len(remaining)}): {remaining}")
    stunden = int(rest_fussweg // 3600)
    minuten = int((rest_fussweg % 3600) // 60)
    print(f"Restfußweg: {rest_fussweg} s  ({stunden} h {minuten} min)")

**Zentrale Einstiegsfunktion, die den gesamten Ablauf zusammenführt.** <br>

**solve:** lädt die Daten, bestimmt alle Maschinen, bildet die Cluster, optimiert über tune_weights das Gewicht und gibt die Lösung aus → Top-Level-Funktion, die alle Bausteine in der richtigen Reihenfolge verbindet

In [81]:
def solve():
    points, matrix = transform_data()
    all_machines = [index for index in points if index != 0]
    clusters = cluster_machines(points)
    tours, abdeckung, remaining, rest_fussweg, best_w = tune_weights(clusters, all_machines, matrix)
    print_solution(tours, abdeckung, remaining, rest_fussweg, best_w)
    return tours, abdeckung, remaining, rest_fussweg, best_w

_ = solve()

Beste Gewichtung w: 0.2
Abgedeckte Maschinen: 45
Roboter 1: [0, 13, 54, 26, 9, 24, 23, 50, 29, 4, 1, 34, 8, 30, 20, 51, 0]
Roboter 2: [0, 52, 39, 46, 53, 28, 33, 3, 22, 15, 37, 41, 16, 47, 6, 14, 0]
Roboter 3: [0, 43, 45, 25, 5, 49, 40, 31, 27, 48, 35, 36, 12, 44, 11, 32, 0]
Übrige Maschinen (9): [2, 7, 10, 17, 18, 19, 21, 38, 42]
Restfußweg: 23850.636899328587 s  (6 h 37 min)
